# 02 · ¿Cambia la prima por cadena, formato o región?

**Objetivo:** Medir con precios reales cuánto sobreprecio sostiene la marca líder frente a su competidor directo, como caso de valor de marca para directores de marketing y ventas.

**Pregunta:** ¿En qué cadenas, formatos y regiones sostiene Coca-Cola más sobreprecio sobre Pepsi?

Importa porque el precio de anaquel lo fija la cadena: si la prima cambia mucho entre cadenas, la marca negocia
distinto con cada una (o cada cadena usa a una de las dos marcas como gancho). Grupos con 10 pares o más.

In [1]:
import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_v, "1")
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import entorno
entorno.setup()
import profeco, tablas, pares, estadistica as E, charts as C
import prima as P
PROC, META, CHARTS, SOCIAL = entorno.PROC, entorno.META, entorno.CHARTS, entorno.SOCIAL
CFG = entorno.config()
reg = entorno.registrar
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 70)
pts, st, cat = P.load()
REPS = 500
out = {}
for k in ["lata", "familiar"]:
    p = pd.read_csv(PROC / f"pares_{k}.csv")
    _, des = P.summary(p, reps=REPS)
    des["comparacion"] = k
    out[k] = des
    print(P.COMPARA[k]["nombre"])
    display(des[["por", "grupo", "n_pares", "municipios", "brecha_pct", "ic90_inf", "ic90_sup", "brecha_pesos",
                 "pct_invertidos", "solido"]].round(1))

Lata 355 ml vs. lata 355 ml


,por,grupo,n_pares,municipios,brecha_pct,ic90_inf,ic90_sup,brecha_pesos,pct_invertidos,solido
0,cadena_comercial,Bodega Aurrera,35,30,26.3,23.7,26.3,5.0,0.0,True
1,cadena_comercial,Chedraui,10,10,24.3,18.4,26.3,4.6,0.0,True
2,cadena_comercial,Hipermercado Soriana,26,23,19.5,17.2,19.5,4.0,0.0,True
3,cadena_comercial,Mega Soriana,14,13,18.2,15.8,19.5,3.8,0.0,True
4,cadena_comercial,Wal-mart,43,37,23.7,21.1,26.3,4.5,0.0,True
5,cadena_comercial,Wal-mart Express,18,13,26.3,26.3,26.3,5.0,5.6,True
6,formato,Bodega Aurrerá,35,30,26.3,23.7,26.3,5.0,0.0,True
7,formato,Chedraui,10,10,24.3,18.4,26.3,4.6,0.0,True
8,formato,Hipermercado Soriana,26,23,19.5,17.2,19.5,4.0,0.0,True
9,formato,Mega Soriana,14,13,18.2,15.8,19.5,3.8,0.0,True


Coca-Cola 2.75 L vs. Pepsi 3 L


,por,grupo,n_pares,municipios,brecha_pct,ic90_inf,ic90_sup,brecha_pesos,pct_invertidos,solido
0,cadena_comercial,Bodega Aurrera,27,22,-5.6,-5.6,0.2,-7.0,63.0,False
1,cadena_comercial,Chedraui,13,13,10.3,7.9,16.0,0.5,0.0,True
2,cadena_comercial,Hipermercado Soriana,17,15,4.6,4.6,6.0,-2.0,11.8,True
3,cadena_comercial,Mega Soriana,15,14,4.6,-1.8,6.0,-2.0,33.3,False
4,cadena_comercial,Wal-mart,30,24,4.9,4.9,5.9,-2.0,0.0,True
5,cadena_comercial,Wal-mart Express,17,12,4.9,4.9,4.9,-2.0,0.0,True
6,formato,Bodega Aurrerá,27,22,-5.6,-5.6,0.2,-7.0,63.0,False
7,formato,Chedraui,13,13,10.3,7.9,16.0,0.5,0.0,True
8,formato,Hipermercado Soriana,17,15,4.6,4.6,6.0,-2.0,11.8,True
9,formato,Mega Soriana,15,14,4.6,-1.8,6.0,-2.0,33.3,False


**Lectura.** En la lata todas las cadenas y regiones sostienen una prima clara. En la botella familiar el
desglose separa: en algunas cadenas Coca-Cola sale más barata por litro.

In [2]:
des = pd.concat(out.values())
des.to_csv(PROC / "prima_desglose.csv", index=False, encoding="utf-8-sig")
F = "02_cadena_formato_region"
lf = out["lata"][out["lata"].por == "formato"].set_index("grupo")
ff = out["familiar"][out["familiar"].por == "formato"].set_index("grupo")
lr = out["lata"][out["lata"].por == "region"].set_index("grupo")
reg("lata_formato_max", lf.brecha_pct.max(), fuente=F, nota=lf.brecha_pct.idxmax())
reg("lata_formato_min", lf.brecha_pct.min(), fuente=F, nota=lf.brecha_pct.idxmin())
reg("lata_formatos", len(lf), fuente=F); reg("lata_formatos_solidos", int(lf.solido.sum()), fuente=F)
reg("lata_region_min", lr.brecha_pct.min(), fuente=F, nota=lr.brecha_pct.idxmin()); reg("lata_region_max", lr.brecha_pct.max(), fuente=F, nota=lr.brecha_pct.idxmax())
for f_, k_ in {"Walmart": "walmart", "Bodega Aurrerá": "bodega", "Hipermercado Soriana": "soriana", "Chedraui": "chedraui"}.items():
    if f_ in lf.index:
        reg(f"lata_{k_}", lf.loc[f_, "brecha_pct"], [lf.loc[f_, "ic90_inf"], lf.loc[f_, "ic90_sup"]], n=lf.loc[f_, "n_pares"], fuente=F)
    if f_ in ff.index:
        reg(f"fam_{k_}", ff.loc[f_, "brecha_pct"], [ff.loc[f_, "ic90_inf"], ff.loc[f_, "ic90_sup"]], n=ff.loc[f_, "n_pares"], fuente=F)
reg("fam_formatos_neg", int((ff.brecha_pct < 0).sum()), n=len(ff), fuente=F)
display(Markdown(
    f"**Conclusión.** En la lata, la prima va de {lf.brecha_pct.min():.1f}% en {lf.brecha_pct.idxmin()} a {lf.brecha_pct.max():.1f}% en "
    f"{lf.brecha_pct.idxmax()}, con rango probable arriba de 0 en {int(lf.solido.sum())} de {len(lf)} formatos. En la botella familiar, "
    f"{int((ff.brecha_pct < 0).sum())} de {len(ff)} formatos tienen la Coca-Cola más barata por litro que la Pepsi."))

**Conclusión.** En la lata, la prima va de 18.2% en Mega Soriana a 26.3% en Bodega Aurrerá, con rango probable arriba de 0 en 6 de 6 formatos. En la botella familiar, 1 de 6 formatos tienen la Coca-Cola más barata por litro que la Pepsi.